# MiniCells HybridCLM v0.2.0a1 Release Publisher

This notebook checks out an immutable release tag, exports and validates the engineering Cell mutation, renders Model/Space assets, and publishes to Hugging Face only after validation.

Scientific status: **Engineering Evidence · Formal Validation Pending**. Formal seeds are never executed. Set `PUBLISH=True` only after reviewing the dry run.

In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys

REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
RELEASE_TAG = 'v0.2.0a1'
WORKTREE = Path('/kaggle/working/mini-cells')
PUBLISH = False
HF_MODEL_REPO = 'archelabs-org/granite-3.1-1b-hybrid-cell-l7-k64'
HF_SPACE_REPO = 'archelabs-org/MiniCells-HybridCLM'


In [ ]:
import shutil, torch
print({'python': sys.version, 'platform': platform.platform(), 'cuda': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count()})
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
print({'disk_free_gb': shutil.disk_usage('/kaggle/working').free / 1e9})


In [ ]:
if not WORKTREE.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(WORKTREE)], check=True)
subprocess.run(['git', 'fetch', '--tags', '--force'], cwd=WORKTREE, check=True)
subprocess.run(['git', 'checkout', '--detach', RELEASE_TAG], cwd=WORKTREE, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKTREE, text=True).strip()
config = json.loads((WORKTREE / 'artifacts/releases/hybrid-clm-v0.1/release-config.json').read_text())
if config['source']['commit'].startswith('<') or config['source']['commit'] != commit:
    raise RuntimeError('RELEASE_IDENTITY_MISMATCH: release config is not pinned to this tag')
print({'tag': RELEASE_TAG, 'commit': commit})


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[hybrid]'], cwd=WORKTREE, check=True)
subprocess.run([sys.executable, 'scripts/release/check_release_identity.py', '--tag', RELEASE_TAG, '--release-config', 'artifacts/releases/hybrid-clm-v0.1/release-config.json'], cwd=WORKTREE, check=True)
subprocess.run([sys.executable, 'scripts/release/export_hybrid_clm_mutation.py', '--release-config', 'artifacts/releases/hybrid-clm-v0.1/release-config.json', '--output', '/kaggle/working/release-output/mutation'], cwd=WORKTREE, check=True)
subprocess.run([sys.executable, 'scripts/release/validate_hybrid_clm_publication.py', '--release-config', 'artifacts/releases/hybrid-clm-v0.1/release-config.json', '--mutation', '/kaggle/working/release-output/mutation', '--output', '/kaggle/working/release-output/PUBLICATION_VALIDATION.json', '--device', 'cuda'], cwd=WORKTREE, check=True)
subprocess.run([sys.executable, 'scripts/release/prepare_hybrid_clm_release.py', '/kaggle/working/release-output/mutation', '--release-config', 'artifacts/releases/hybrid-clm-v0.1/release-config.json', '--validation-report', '/kaggle/working/release-output/PUBLICATION_VALIDATION.json', '--output', '/kaggle/working/release-output/hf-model'], cwd=WORKTREE, check=True)
subprocess.run([sys.executable, 'scripts/release/render_hybrid_clm_release_assets.py', '--release-config', 'artifacts/releases/hybrid-clm-v0.1/release-config.json', '--output', '/kaggle/working/release-output/hf-space'], cwd=WORKTREE, check=True)


In [ ]:
report = json.loads(Path('/kaggle/working/release-output/PUBLICATION_VALIDATION.json').read_text())
if report['status'] != 'READY_FOR_PUBLICATION':
    raise RuntimeError('publication validation did not pass')
publication = {'status': 'DRY_RUN', 'release': report, 'huggingface': {'model_repo': HF_MODEL_REPO, 'space_repo': HF_SPACE_REPO, 'collection': 'MiniCells — Hybrid CLM'}, 'formal_execution_started': False, 'formal_seeds_untouched': True}
if PUBLISH:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import EntryNotFoundError, HfApi, RepositoryNotFoundError, hf_hub_download
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if not token:
        raise RuntimeError('HF_AUTH_FAILED: HF_TOKEN is not configured')
    api = HfApi(token=token)
    def upload_if_compatible(repo_id, repo_type, folder, marker):
        try:
            remote = Path(hf_hub_download(repo_id, marker, repo_type=repo_type, token=token))
        except EntryNotFoundError:
            remote = None
        if remote is not None:
            payload = json.loads(remote.read_text())
            release = payload.get('release', {})
            if payload.get('release_tag') != RELEASE_TAG and release.get('tag') != RELEASE_TAG:
                raise RuntimeError('HF_PUBLICATION_CONFLICT: existing repository is a different release')
            return 'UNCHANGED'
        try:
            api.repo_info(repo_id, repo_type=repo_type)
        except RepositoryNotFoundError:
            pass
        else:
            raise RuntimeError('HF_PUBLICATION_CONFLICT: existing repository lacks a compatible release marker')
        api.create_repo(repo_id, repo_type=repo_type, space_sdk='gradio' if repo_type == 'space' else None, exist_ok=True)
        api.upload_folder(repo_id=repo_id, repo_type=repo_type, folder_path=folder, commit_message=RELEASE_TAG)
        return 'UPDATED'
    publication['huggingface']['model_status'] = upload_if_compatible(HF_MODEL_REPO, 'model', '/kaggle/working/release-output/hf-model', 'provenance.json')
    publication['huggingface']['space_status'] = upload_if_compatible(HF_SPACE_REPO, 'space', '/kaggle/working/release-output/hf-space', 'space-data.json')
    collection = api.create_collection(title='MiniCells — Hybrid CLM', namespace='archelabs-org', exists_ok=True)
    publication['huggingface']['collection'] = collection.slug
    for item_id, item_type in ((HF_MODEL_REPO, 'model'), (HF_SPACE_REPO, 'space')):
        api.add_collection_item(collection_slug=collection.slug, item_id=item_id, item_type=item_type)
    publication['status'] = 'PUBLICATION_COMPLETE'
Path('/kaggle/working/release-output/HF_PUBLICATION_REPORT.json').write_text(json.dumps(publication, indent=2))
print('MINICELLS_HF_RELEASE=' + ('PASS' if PUBLISH else 'DRY_RUN'))
print('HF_MODEL=' + HF_MODEL_REPO)
print('HF_SPACE=' + HF_SPACE_REPO)
